In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv("/content/anime.csv")

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

Dataset Shape: (12294, 7)

First 5 Rows:


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB

Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Statistical Summary:


,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [75]:
# Make a copy
anime = df.copy()

# -----------------------------
# Handle missing values
# -----------------------------

anime["genre"] = anime["genre"].fillna("")

anime["type"] = anime["type"].fillna(
    anime["type"].mode()[0]
)

anime["rating"] = anime["rating"].fillna(
    anime["rating"].median()
)

# -----------------------------
# Clean episodes
# -----------------------------

anime["episodes"] = pd.to_numeric(
    anime["episodes"],
    errors="coerce"
)

anime["episodes"] = anime["episodes"].fillna(
    anime["episodes"].median()
)

print("Missing Values After Preprocessing:")
print(anime.isnull().sum())

# -----------------------------
# Convert genre to TF-IDF
# -----------------------------

tfidf = TfidfVectorizer(
    stop_words="english"
)

genre_matrix = tfidf.fit_transform(
    anime["genre"]
)

print("\nGenre Matrix Shape:", genre_matrix.shape)

# -----------------------------
# Encode anime type
# -----------------------------

type_dummies = pd.get_dummies(
    anime["type"],
    prefix="type"
)

# -----------------------------
# Scale numerical features
# -----------------------------

numeric_features = anime[
    ["episodes", "rating", "members"]
]

scaler = StandardScaler()

numeric_matrix = scaler.fit_transform(
    numeric_features
)

print("Numerical Features Scaled Successfully.")

Missing Values After Preprocessing:
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

Genre Matrix Shape: (12294, 46)
Numerical Features Scaled Successfully.


In [76]:
from scipy.sparse import hstack, csr_matrix

# Convert type encoding to sparse matrix
type_matrix = csr_matrix(
    type_dummies.astype(float).values
)

# Convert numerical features to sparse matrix
numeric_sparse = csr_matrix(
    numeric_matrix
)

# Combine all features
feature_matrix = hstack([
    genre_matrix,
    type_matrix,
    numeric_sparse
])

print(
    "Combined Feature Matrix Shape:",
    feature_matrix.shape
)

# Calculate cosine similarity
cosine_sim = cosine_similarity(
    feature_matrix,
    feature_matrix
)

print(
    "Cosine Similarity Matrix Shape:",
    cosine_sim.shape
)

Combined Feature Matrix Shape: (12294, 55)
Cosine Similarity Matrix Shape: (12294, 12294)


In [77]:
# Create title-to-index mapping
anime_indices = pd.Series(
    anime.index,
    index=anime["name"].str.lower()
).drop_duplicates()


def recommend_anime(
    anime_title,
    top_n=10,
    threshold=0.30
):

    title = anime_title.lower()

    if title not in anime_indices:
        print("Anime not found in dataset.")
        return

    idx = anime_indices[title]

    # Get similarity scores
    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    # Sort by similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores[1:]:

        if score >= threshold:

            recommendations.append({
                "Anime": anime.iloc[i]["name"],
                "Similarity Score": round(score, 4)
            })

        if len(recommendations) >= top_n:
            break

    return pd.DataFrame(
        recommendations
    )


# Example recommendation
target_anime = anime.iloc[0]["name"]

print("Selected Anime:", target_anime)

recommendations = recommend_anime(
    target_anime,
    top_n=10,
    threshold=0.30
)

display(recommendations)

Selected Anime: Kimi no Na wa.


,Anime,Similarity Score
0,Hotarubi no Mori e,0.9749
1,Suzumiya Haruhi no Shoushitsu,0.9660
2,Hotaru no Haka,0.9529
3,Majo no Takkyuubin,0.9495
4,Kotonoha no Niwa,0.9483
5,Tenkuu no Shiro Laputa,0.9431
6,Ookami Kodomo no Ame to Yuki,0.9424
7,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,0.9389
8,Evangelion: 2.0 You Can (Not) Advance,0.9366
9,Neon Genesis Evangelion: The End of Evangelion,0.9362


In [78]:
print("Threshold Experiment:\n")

for threshold in [0.20, 0.30, 0.40, 0.50]:

    result = recommend_anime(
        target_anime,
        top_n=10,
        threshold=threshold
    )

    count = 0 if result is None else len(result)

    print(
        f"Threshold = {threshold} "
        f"-> {count} recommendations"
    )

Threshold Experiment:

Threshold = 0.2 -> 10 recommendations
Threshold = 0.3 -> 10 recommendations
Threshold = 0.4 -> 10 recommendations
Threshold = 0.5 -> 10 recommendations


# Analysis and Conclusion

## Recommendation System Analysis

The recommendation system uses content-based similarity to recommend anime
titles with similar characteristics.

Genre information is converted into TF-IDF features because an anime can
belong to multiple genres. Anime type is converted using one-hot encoding,
while numerical variables such as episodes, rating, and members are
standardized.

Cosine similarity is then calculated between the feature vectors of all
anime titles.

A similarity threshold controls the strictness of the recommendations.
A lower threshold produces more recommendations but may include less similar
anime. A higher threshold produces fewer recommendations with stronger
similarity.

## Areas for Improvement

The recommendation system can be improved by:

1. Using user viewing or rating history.
2. Implementing collaborative filtering.
3. Combining content-based and collaborative filtering.
4. Using additional anime features.
5. Using user-specific recommendations instead of general content similarity.

## Conclusion

A content-based anime recommendation system was successfully implemented
using cosine similarity.

The system preprocesses missing values, converts categorical information into
numerical representations, scales numerical features, and calculates
similarity between anime titles.

The system can recommend anime similar to a selected title. Different
similarity thresholds were also tested to understand their effect on the
number of recommendations.

The main limitation is that the system recommends anime based on item
characteristics and does not consider individual user preferences.

## Interview Questions

### 1. What is the difference between user-based and item-based collaborative filtering?

User-based collaborative filtering recommends items based on the preferences
of users who have similar interests.

Item-based collaborative filtering recommends items that are similar to items
that a particular user has already liked or interacted with.

For example, if two users have similar anime preferences, user-based
collaborative filtering can recommend anime liked by one user to the other.
Item-based collaborative filtering instead finds anime similar to anime that
the user has already watched or rated.

### 2. What is collaborative filtering, and how does it work?

Collaborative filtering is a recommendation technique that uses the behavior
and preferences of users to make recommendations.

It analyzes information such as ratings, purchases, views, or interactions
and identifies similarities between users or items.

It can be divided into:

- User-based collaborative filtering
- Item-based collaborative filtering

Unlike content-based filtering, collaborative filtering does not primarily
depend on the characteristics of the item. Instead, it learns from user-item
interactions.